In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import math
import numpy as np
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
import math
import seaborn as sns
from sklearn.datasets import (
    make_blobs,
    make_moons,
    make_swiss_roll,
    make_circles,
    make_s_curve
)
sns.set_style("whitegrid")
import os
import shutil
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR, LambdaLR
import random
!pip install -U wandb

import wandb
wandb.login(key="wandb_api_key")

# Models : 

In [ ]:
# ---------------------------------------------------------------------------
# Positional / Condition Embeddings
# ---------------------------------------------------------------------------

class SinusoidalPositionalEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class EmbeddingBlock(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.sequence = nn.Sequential(
            SinusoidalPositionalEmbeddings(input_dim),
            nn.Linear(input_dim, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, x):
        return self.sequence(x)


# ---------------------------------------------------------------------------
# AdaIN  (unchanged from original)
# ---------------------------------------------------------------------------

class AdaIN(nn.Module):
    """Adaptive Normalization – injects condition via scale + shift."""

    def __init__(self, feature_channels, condition_dim):
        super().__init__()
        # self.norm = nn.InstanceNorm2d(feature_channels, affine=False) ## TODO
        self.norm = nn.GroupNorm(min(8, feature_channels), feature_channels, affine=False)

        self.fc   = nn.Linear(condition_dim, feature_channels * 2)

    def forward(self, x, condition):
        h = self.fc(condition).view(condition.size(0), -1, 1, 1)
        gamma, beta = torch.chunk(h, 2, dim=1)
        return gamma * self.norm(x) + beta


# ---------------------------------------------------------------------------
# Residual Block with double AdaIN conditioning
# ---------------------------------------------------------------------------

class ResBlock(nn.Module):
    """
    Two conv layers, each followed by AdaIN + SiLU.
    A 1x1 shortcut handles channel mismatch.
    Doubles capacity vs. a single conv while keeping gradient flow healthy —
    important for score-matching losses in Schrödinger Bridge training.
    """

    def __init__(self, in_channels, out_channels, condition_dim, dropout_rate=0.1):
        super().__init__()
        self.conv1   = nn.Conv2d(in_channels,  out_channels, 3, padding=1)
        self.adain1  = AdaIN(out_channels, condition_dim)
        self.dropout = nn.Dropout2d(dropout_rate)
        self.conv2   = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.adain2  = AdaIN(out_channels, condition_dim)

        # Shortcut: align channels if needed
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, 1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, cond):
        h = F.silu(self.adain1(self.conv1(x), cond))
        h = self.dropout(h)
        h = self.adain2(self.conv2(h), cond)        # no activation yet
        return F.silu(h + self.shortcut(x))         # residual add then activate


# ---------------------------------------------------------------------------
# Self-Attention at the bottleneck
# ---------------------------------------------------------------------------

class SelfAttention2d(nn.Module):
    """
    Multi-head self-attention over spatial positions.
    Placed at the 8×8 bottleneck so the model can reason about *global*
    relationships — critical for learning the Schrödinger Bridge coupling
    between two distributions.
    """

    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)          # stable across batch sizes
        self.attn = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H * W).transpose(1, 2)   # [B, HW, C]
        h, _ = self.attn(h, h, h)
        h = h.transpose(1, 2).reshape(B, C, H, W)
        return x + h      # residual


# ---------------------------------------------------------------------------
# Improved UNet  (~5.1 M parameters with base_channels=128)
# ---------------------------------------------------------------------------

class Enhanced_UNet(nn.Module):
    """
    Changes vs. SimpleUNet
    ──────────────────────
    • ResBlocks  instead of bare Conv2d  → richer features, stable gradients
    • AdaIN in *every* conv of every block (was missing in some decoder blocks)
    • Self-attention at the bottleneck   → global context for SB score network
    • base_channels=128 (c1=128, c2=256) as requested

    Parameter budget (approximate, base_channels=128, condition_dim=64)
    ────────────────────────────────────────────────────────────────────
      enc1_block   (  1 → 128)  ~  182 K
      enc2_block   (128 → 256)  ~  984 K
      bottleneck   (256 → 256)  ~ 1246 K
      attention    (256)        ~  262 K
      dec1_block   (384 → 256)  ~ 1639 K   (skip: 128+256 concat)
      dec2_block   (256 → 128)  ~  508 K   (skip: 128+128 concat)
      up-convs × 2              ~  262 K
      embeddings + final        ~   18 K
      ─────────────────────────────────
      TOTAL                     ~  5.10 M
    """

    def __init__(
        self,
        time_embed_dim: int = 32,
        dir_embed_dim:  int = 32,
        base_channels:  int = 128,
        dropout_rate:   float = 0.1,
    ):
        super().__init__()

        c1 = base_channels        # 128
        c2 = base_channels * 2    # 256
        condition_dim = time_embed_dim + dir_embed_dim   # 64

        # ── Condition embeddings ───────────────────────────────────────────
        self.time_embed = EmbeddingBlock(input_dim=time_embed_dim, embed_dim=time_embed_dim)
        
        # self.dir_embed  = EmbeddingBlock(input_dim=dir_embed_dim,  embed_dim=dir_embed_dim)
        self.dir_embed = nn.Sequential(
                nn.Linear(1, dir_embed_dim),
                nn.SiLU(),
                nn.Linear(dir_embed_dim, dir_embed_dim),
            )
                    
        # ── Encoder ───────────────────────────────────────────────────────
        # 32×32 → 32×32
        self.enc1_block = ResBlock(1,  c1, condition_dim, dropout_rate)
        self.pool1      = nn.MaxPool2d(2, 2)

        # 16×16 → 16×16
        self.enc2_block = ResBlock(c1, c2, condition_dim, dropout_rate)
        self.pool2      = nn.MaxPool2d(2, 2)

        # ── Bottleneck  (8×8) ─────────────────────────────────────────────
        self.bottleneck = ResBlock(c2, c2, condition_dim, dropout_rate)
        self.attn       = SelfAttention2d(c2, num_heads=4)   # global context

        # ── Decoder ───────────────────────────────────────────────────────
        # 8→16
        self.up1        = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        # skip from enc2 (c2) + up1 (c1) → c1+c2 = 384 in
        self.dec1_block = ResBlock(c1 + c2, c2, condition_dim, dropout_rate)

        # 16→32
        self.up2        = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        # skip from enc1 (c1) + up2 (c1) → c1+c1 = 256 in
        self.dec2_block = ResBlock(c1 + c1, c1, condition_dim, dropout_rate)

        # ── Output ────────────────────────────────────────────────────────
        self.final_conv = nn.Conv2d(c1, 1, kernel_size=3, padding=1)

    # ----------------------------------------------------------------------
    def forward(self, x_vector, t, s):
        # 1. Build condition vector
        if t.dim() == 1: t = t.unsqueeze(-1)
        if s.dim() == 1: s = s.unsqueeze(-1)
        t_emb = self.time_embed(t.squeeze(-1))
        
        # s_emb = self.dir_embed(s.squeeze(-1)) # TODO
        s_emb = self.dir_embed(s)
        
        cond  = torch.cat([t_emb, s_emb], dim=1)   # [B, condition_dim]

        # 2. Reshape flat vector → spatial map  [B, 1, 32, 32]
        x = x_vector.reshape(x_vector.shape[0], 1, 32, 32)

        # ── Encoder ──
        h1 = self.enc1_block(x,        cond)   # [B, c1, 32, 32]
        h2 = self.enc2_block(self.pool1(h1), cond)   # [B, c2, 16, 16]
        hb = self.bottleneck(self.pool2(h2), cond)   # [B, c2,  8,  8]
        hb = self.attn(hb)                           # global self-attention

        # ── Decoder ──
        d1 = self.up1(hb)                            # [B, c1, 16, 16]
        d1 = self.dec1_block(torch.cat([d1, h2], 1), cond)  # [B, c2, 16, 16]

        d2 = self.up2(d1)                            # [B, c1, 32, 32]
        d2 = self.dec2_block(torch.cat([d2, h1], 1), cond)  # [B, c1, 32, 32]

        out = self.final_conv(d2)                    # [B,  1, 32, 32]
        return out.reshape(x_vector.shape[0], -1)
        
class SimpleUNet(nn.Module):
    def __init__(self, time_embed_dim=32, dir_embed_dim=32, base_channels=64, dropout_rate=0.1):
        super(SimpleUNet, self).__init__()

        # --- Configurations ---
        c1 = base_channels       # Level 1 channels
        c2 = base_channels * 2   # Level 2 channels
        
        # --- Embeddings ---
        self.time_embed = EmbeddingBlock(input_dim=time_embed_dim, embed_dim=time_embed_dim)
        # self.dir_embed = EmbeddingBlock(input_dim=dir_embed_dim, embed_dim=dir_embed_dim) # TODO
        self.dir_embed = nn.Sequential(
                nn.Linear(1, dir_embed_dim),
                nn.SiLU(),
                nn.Linear(dir_embed_dim, dir_embed_dim),
            )
        
        condition_dim = time_embed_dim + dir_embed_dim

        self.dropout = nn.Dropout(dropout_rate)

        # --- ENCODER ---
        # Block 1: Input (1) -> c1
        self.enc1_conv = nn.Conv2d(1, c1, kernel_size=3, padding=1)
        self.enc1_adain = AdaIN(c1, condition_dim)
        
        # Block 2: c1 -> c2
        self.enc2_conv = nn.Conv2d(c1, c2, kernel_size=3, padding=1)
        self.enc2_adain = AdaIN(c2, condition_dim)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # --- DECODER ---
        # Up 1: c2 (Bottleneck) -> c1 (spatial 8->16)
        self.up1 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        
        # Block 3 (Decoder 1): Input (c1 from up + c2 from skip) -> c2
        self.dec1_conv = nn.Conv2d(c1 + c2, c2, kernel_size=3, padding=1)
        self.dec1_adain = AdaIN(c2, condition_dim) # FIXED: Added AdaIN

        # Up 2: c2 -> c1 (spatial 16->32)
        self.up2 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        
        # Block 4 (Decoder 2): Input (c1 from up + c1 from skip) -> c1
        self.dec2_conv = nn.Conv2d(c1 + c1, c1, kernel_size=3, padding=1)
        self.dec2_adain = AdaIN(c1, condition_dim) # FIXED: Added AdaIN

        # Final: c1 -> 1
        self.final_conv = nn.Conv2d(c1, 1, kernel_size=3, padding=1)

    def forward(self, x_vector, t, s):
        # 1. Embeddings
        if t.dim() == 1: t = t.unsqueeze(-1)
        if s.dim() == 1: s = s.unsqueeze(-1)
        t_emb = self.time_embed(t.squeeze(-1))
        # s_emb = self.dir_embed(s.squeeze(-1)) # TODO
        s_emb = self.dir_embed(s)  

        cond = torch.cat([t_emb, s_emb], dim=1)

        # 2. Reshape [B, D] -> [B, 1, 32, 32]
        x = x_vector.reshape(x_vector.shape[0], 1, 32, 32)

        # --- Encoder ---
        # Level 1
        h1 = self.enc1_conv(x)          # [B, c1, 32, 32]
        h1 = self.enc1_adain(h1, cond)
        h1 = F.leaky_relu(h1)
        h1 = self.dropout(h1)
        
        h2_in = self.pool(h1)           # [B, c1, 16, 16]

        # Level 2
        h2 = self.enc2_conv(h2_in)      # [B, c2, 16, 16]
        h2 = self.enc2_adain(h2, cond)
        h2 = F.leaky_relu(h2)
        h2 = self.dropout(h2)
        
        h_bot = self.pool(h2)           # [B, c2, 8, 8] (Bottleneck)

        # --- Decoder ---
        # Level 1 Upsample
        d1 = self.up1(h_bot)            # [B, c1, 16, 16]
        
        # Concat with Enc2 Skip (h2)
        d1 = torch.cat([d1, h2], dim=1) # [B, c1+c2, 16, 16]
        
        d1 = self.dec1_conv(d1)         # [B, c2, 16, 16]
        d1 = self.dec1_adain(d1, cond)  # Inject Condition
        d1 = F.leaky_relu(d1)
        d1 = self.dropout(d1)

        # Level 2 Upsample
        d2 = self.up2(d1)               # [B, c1, 32, 32]
        
        # Concat with Enc1 Skip (h1)
        d2 = torch.cat([d2, h1], dim=1) # [B, c1+c1, 32, 32]
        
        d2 = self.dec2_conv(d2)         # [B, c1, 32, 32]
        d2 = self.dec2_adain(d2, cond)  # Inject Condition
        d2 = F.leaky_relu(d2)

        # Final
        out = self.final_conv(d2)
        return out.reshape(x_vector.shape[0], -1)

class BidirectionalMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, time_embed_dim=32, dir_embed_dim=16):
        super(BidirectionalMLP, self).__init__()

        self.time_embed = nn.Sequential(
            nn.Linear(1, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )
        self.dir_embed = nn.Embedding(2, dir_embed_dim)

        self.net = nn.Sequential(
            nn.Linear(input_dim + time_embed_dim + dir_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x, t, s):
        if t.dim() == 1:
            t = t.unsqueeze(-1)
        if s.dim() == 2:
            s = s.squeeze(-1)

        t_emb = self.time_embed(t)
        s_emb = self.dir_embed(s.long())

        h = torch.cat([x, t_emb, s_emb], dim=1)
        return self.net(h)

# MAIN CLASS 

In [ ]:
class Law_class(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]
        
class SBScheduler:
    def __init__(self, total_steps, ratio=0.4, start_lambda=0.0):
        self.total_steps = total_steps
        self.step = 0
        self.start_lambda = start_lambda
        self.snap_step = int(total_steps * ratio)

    def get_lambda(self):
        if self.step >= self.snap_step:
            self.step += 1
            return 1.0
        # Progress ratio from 0 to 1
        progress = min(1.0, self.step / self.snap_step)
        # Cosine schedule: starts at 0, ends at 1
        cosine_val = 0.5 * (1 - math.cos(math.pi * progress))
        # Linear interpolation between start_lambda and 1.0
        lam = self.start_lambda + (1.0 - self.start_lambda) * cosine_val
        self.step += 1
        return lam

class OscillatingSnapScheduler:
    def __init__(self, total_steps, ratio=0.6, freq=20.0, decay=5.0, device='cpu', phi=None):
        self.total_steps = total_steps
        self.snap_step = int(total_steps * ratio)
        self.freq = freq
        self.decay = decay
        self.device = device
        self.current_step = 0
        
        self.phi = phi if phi is not None else torch.rand(1, device=self.device) * 2 * torch.pi
        
    def step(self):
        """Returns alpha and beta, converging both to 0.5."""
        if self.current_step >= self.snap_step:
            alpha = torch.tensor(0.5, device=self.device)
        else:
            tau = self.current_step / self.snap_step
            # Damping factor
            amplitude = 0.5 * torch.exp(torch.tensor(-self.decay * tau, device=self.device))
            
            # Oscillate around 0.5
            alpha = 0.5 + amplitude * torch.sin(
                2 * torch.pi * self.freq * tau + self.phi
            )
            alpha = torch.clamp(alpha, 0.0, 1.0)
            
        self.current_step += 1
        beta = 1.0 - alpha
        
        return alpha, beta
        
class Schrodinger_Bridge_Matching(object):
    def __init__(self, Law_0, Law_1, pretrain_epochs, finetune_epochs, Bs, steps, eps,
                 pretrain_lr, finetune_lr, decay, base_channels = 64 , use_ema_for_sampling=True, num_workers=0, ratio_osc = 0.6, ratio_leap = 0.3, eps_time = 0.001, warmup_ratio = 0.05):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print('device' , self.device)

        # Convert to tensor if needed
        Law_0_tensor = torch.as_tensor(Law_0, dtype=torch.float32)
        Law_1_tensor = torch.as_tensor(Law_1, dtype=torch.float32)

        self.Law_0 = Law_class(Law_0_tensor)
        self.Law_1 = Law_class(Law_1_tensor)

        self.dim = Law_0_tensor.shape[-1]

        self.T = 1.0
        self.pretrain_epochs = pretrain_epochs
        self.finetune_epochs = finetune_epochs
        self.criterion = nn.MSELoss()

        self.Bs = Bs
        self.bs = int(Bs / 2)
        self.num_workers = num_workers

        self.steps = steps
        self.global_step = 0
        self.eps = eps
        self.pretrain_lr = pretrain_lr
        self.finetune_lr = finetune_lr
        self.decay = decay

        self.eps_time = eps_time
        self.ratio_osc = ratio_osc
        self.ratio_leap = ratio_leap
        self.warmup_ratio = warmup_ratio
        
        self.delta_t = self.T / self.steps
        self.n_steps = int(round(self.T / self.delta_t))
        self.t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, self.n_steps + 1)]

        self.v_theta = Enhanced_UNet(base_channels=base_channels).to(self.device)

        os.makedirs("checkpoints/pretrain", exist_ok=True)
        os.makedirs("checkpoints/finetune", exist_ok=True)
        
        self.v_theta = self.v_theta.to(self.device)

        self.ema_model = copy.deepcopy(self.v_theta).to(self.device)
        self.ema_model.eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)
            

        self.loss_history = {
            'pretrain_total': [],
            'pretrain_forward': [],
            'pretrain_backward': [],
            'finetune_total': [],
            'finetune_forward': [],
            'finetune_backward': []
        }

    def update_ema(self, decay):
        with torch.no_grad():
            for param, ema_param in zip(self.v_theta.parameters(), self.ema_model.parameters()):
                ema_param.mul_(decay).add_(param.data, alpha=1 - decay)

    def get_ema_model(self):
        return self.ema_model  

    def _make_loaders(self, epoch, batch_size , pretrain = True):
        offset_0 = 42 if pretrain else 75
        g = torch.Generator()
        g.manual_seed(offset_0 + epoch)   # epoch-dependent seed 
    
        def worker_init_fn(worker_id):
            np.random.seed(offset_0 + epoch * 1000 + worker_id)
            random.seed(offset_0 + epoch * 1000 + worker_id)
    
        loader_0 = DataLoader(self.Law_0, batch_size=batch_size, shuffle=True,
                              generator=g, worker_init_fn=worker_init_fn,
                              num_workers=self.num_workers, pin_memory=True,drop_last =True)
        
        # independent shuffle
        offset_1 = 99 if pretrain else 156
        g1 = torch.Generator()
        g1.manual_seed(offset_1 + epoch)
        loader_1 = DataLoader(self.Law_1, batch_size=batch_size, shuffle=True,
                              generator=g1, worker_init_fn=worker_init_fn,
                              num_workers=self.num_workers, pin_memory=True, drop_last =True)
        return loader_0, loader_1
    
    def pretrain_bridge(self, fixed_test_batch= None, print_every=500, save_every_epoch = 1, resume_from=None):
        self.v_theta.train()
        optimizer = torch.optim.Adam(self.v_theta.parameters(), lr=self.pretrain_lr)
        
        start_epoch = 0
        if resume_from is not None:
            start_epoch = self.load_checkpoint(resume_from, optimizer=optimizer, scheduler=None)
        
        forward_dir = torch.ones(self.bs, device=self.device)
        backward_dir = torch.zeros(self.Bs - self.bs, device=self.device)

        for epoch in range(start_epoch, self.pretrain_epochs):
            epoch_loss = 0.0
            epoch_fwd_loss = 0.0
            epoch_bwd_loss = 0.0
            n_batches = 0

            loader_0, loader_1 = self._make_loaders(epoch, self.Bs) 

            pbar = tqdm(zip(loader_0, loader_1), desc=f"Pretrain Epoch {epoch+1}/{self.pretrain_epochs}")

            for j ,(X0, X1) in enumerate(pbar):    
                X0 = X0.to(self.device)
                X1 = X1.to(self.device)

                batch_size = min(X0.shape[0], X1.shape[0])
                X0 = X0[:batch_size]
                X1 = X1[:batch_size]

                # t = torch.rand(batch_size, 1, device=self.device)
                t = self.eps_time  + (1 - 2 * self.eps_time ) * torch.rand(batch_size, 1, device=self.device)
                Z = torch.randn(batch_size, self.dim, device=self.device)

                Xt = (1 - t) * X0 + t * X1 + torch.sqrt(self.eps * t * (1 - t)) * Z

                optimizer.zero_grad()

                bs_actual = batch_size // 2
                v_forward = self.v_theta(Xt[:bs_actual], t[:bs_actual], forward_dir[:bs_actual])
                target_forward = (X1[:bs_actual] - Xt[:bs_actual]) / torch.clamp(1 - t[:bs_actual], min=1e-3)
                forward_loss = self.criterion(v_forward, target_forward)

                v_backward = self.v_theta(Xt[bs_actual:],1.0 - t[bs_actual:], backward_dir[:batch_size-bs_actual])
                target_backward = (X0[bs_actual:] - Xt[bs_actual:]) / torch.clamp(t[bs_actual:], min=1e-3)
                backward_loss = self.criterion(v_backward, target_backward)

                loss = 0.5 * (forward_loss + backward_loss)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.v_theta.parameters(), max_norm=1.0)
                optimizer.step()
                self.update_ema(self.decay)

                epoch_loss += loss.item()
                epoch_fwd_loss += forward_loss.item()
                epoch_bwd_loss += backward_loss.item()
                n_batches += 1

                # Compute norms
                v_forward_norm = v_forward.detach().norm(dim=-1).mean().item()
                v_backward_norm = v_backward.detach().norm(dim=-1).mean().item()

                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'fwd': f'{forward_loss.item():.4f}',
                    'bwd': f'{backward_loss.item():.4f}'
                })

                if wandb.run is not None:
                    wandb.log({
                        "pretrain/loss": loss.item(),
                        "pretrain/fwd_loss": forward_loss.item(),
                        "pretrain/bwd_loss": backward_loss.item(),
                        "pretrain/v_forward_norm": v_forward_norm,
                        "pretrain/v_backward_norm": v_backward_norm,
                        "epoch": epoch
                    }, step=self.global_step)

                    if self.global_step % print_every == 0 and fixed_test_batch is not None :
                        fig = self.plot_progress(fixed_test_batch, title=f"Pretrain Step {self.global_step}")
                        wandb.log({"pretrain/samples": wandb.Image(fig)}, step=self.global_step)

                        os.system("nvidia-smi")
                        
                self.global_step += 1

            if epoch % save_every_epoch == 0 :
                self.save_checkpoint(
                        f"checkpoints/pretrain/step_{self.global_step}.pt",
                        optimizer=optimizer,
                        scheduler=None,
                        epoch=epoch,
                    )

            self.loss_history['pretrain_total'].append(epoch_loss / n_batches)
            self.loss_history['pretrain_forward'].append(epoch_fwd_loss / n_batches)
            self.loss_history['pretrain_backward'].append(epoch_bwd_loss / n_batches)

    def finetune_bridge(self, fixed_test_batch=None, print_every=500, save_every_epoch = 1, resume_from=None):
        
        self.v_theta.train()
        
        optimizer = torch.optim.Adam(self.v_theta.parameters(), lr=self.finetune_lr)
        
        total_steps = self.finetune_epochs * max(len(DataLoader(self.Law_0, batch_size=self.bs)),
                                                  len(DataLoader(self.Law_1, batch_size=self.bs)))
        warmup_steps = int(self.warmup_ratio * total_steps)
        
        warmup = LinearLR(
            optimizer,
            start_factor=0.001,
            end_factor=1.0,
            total_iters=warmup_steps
        )
        # rest = CosineAnnealingLR(
        #     optimizer,
        #     T_max=total_steps - warmup_steps,
        #     eta_min=self.finetune_lr * 0.01
        # )

        rest = LambdaLR(
                optimizer,
                lr_lambda=lambda step: 1.0  # keeps LR constant
            )
        
        scheduler = SequentialLR(optimizer, schedulers=[warmup, rest], milestones=[warmup_steps])
        
        sb_scheduler = SBScheduler(total_steps=total_steps , ratio=self.ratio_leap)
        ratio_scheduler = OscillatingSnapScheduler(total_steps=total_steps, ratio=self.ratio_osc, device=self.device)
        
        start_epoch = 0
        if resume_from is not None:
            start_epoch = self.load_checkpoint(resume_from, optimizer=optimizer, scheduler=scheduler,sb_scheduler = sb_scheduler, ratio_scheduler=ratio_scheduler)

        for epoch in range(start_epoch, self.finetune_epochs):   
            epoch_loss = 0.0
            epoch_fwd_loss = 0.0
            epoch_bwd_loss = 0.0
            n_batches = 0

            loader_0, loader_1 = self._make_loaders(epoch, self.bs, pretrain=False)  # finetune

            pbar = tqdm(zip(loader_0, loader_1), desc=f"Finetune Epoch {epoch+1}/{self.finetune_epochs}")

            for j ,(X0, X1) in enumerate(pbar):
                
                X0 = X0.to(self.device)
                X1 = X1.to(self.device)

                current_bs = min(X0.shape[0], X1.shape[0])
                X0 = X0[:current_bs]
                X1 = X1[:current_bs]                
                fwd_ones = torch.ones(current_bs, device=self.device)
                bwd_zeros = torch.zeros(current_bs, device=self.device)

                # Generate Samples
                sample_model = self.get_ema_model()
                sample_model.eval()
                
                with torch.no_grad():
                    # Generate X1_tilde (Forward: X0 -> X1)
                    X1_tilde = X0.clone()
                    for i , t in enumerate(self.t_list[:-1]):
                        t_tensor = t.expand(current_bs, 1)
                        drift = sample_model(X1_tilde, t_tensor, fwd_ones)
                        # X1_tilde = X1_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X1_tilde)
                        if i < self.n_steps - 1:
                            X1_tilde = X1_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X1_tilde)
                        else:
                            X1_tilde = X1_tilde + self.delta_t * drift
                
                    # Generate X0_tilde (Backward: X1 -> X0)
                    X0_tilde = X1.clone()
                    for i, t in enumerate(reversed(self.t_list[1:])):
                        t_tensor = t.expand(current_bs, 1)
                        drift = sample_model(X0_tilde, 1.0 - t_tensor, bwd_zeros)
                        # X0_tilde = X0_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X0_tilde)
                        if i < self.n_steps - 1:
                            X0_tilde = X0_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X0_tilde)
                        else:
                            X0_tilde = X0_tilde + self.delta_t * drift

                self.v_theta.train()
            
                t_f = self.eps_time  + (1 - 2 * self.eps_time) * torch.rand(current_bs, 1, device=self.device)
                Z_f = torch.randn(current_bs, self.dim, device=self.device)

                t_b = self.eps_time  + (1 - 2 * self.eps_time) * torch.rand(current_bs, 1, device=self.device)
                Z_b = torch.randn(current_bs, self.dim, device=self.device)

                ####### Schedulers ########
                lam = sb_scheduler.get_lambda()
                alpha, beta = ratio_scheduler.step()

                X0_blended = torch.lerp(X0, X0_tilde, lam)
                X1_blended = torch.lerp(X1, X1_tilde, lam)
                
                # Forward & Backward Losses                
                Xt_f = (1 - t_f) * X0_blended + t_f * X1 + torch.sqrt(self.eps * t_f * (1 - t_f)) * Z_f
                Xt_b = (1 - t_b) * X0 + t_b * X1_blended + torch.sqrt(self.eps * t_b * (1 - t_b)) * Z_b
                target_forward = (X1 - Xt_f) / torch.clamp(1 - t_f, min=1e-3) 
                target_backward = (X0 - Xt_b) / torch.clamp(t_b, min=1e-3)

                optimizer.zero_grad()

                # Forward Loss
                v_forward = self.v_theta(Xt_f, t_f, fwd_ones)
                forward_loss = self.criterion(v_forward, target_forward)

                # Backward Loss
                v_backward = self.v_theta(Xt_b, 1.0 - t_b, bwd_zeros)
                backward_loss = self.criterion(v_backward, target_backward)
                
                # loss = 0.5 * (forward_loss + backward_loss)
                loss = alpha * forward_loss + beta * backward_loss
                
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.v_theta.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                self.update_ema(self.decay)
                
                epoch_loss += loss.item()
                epoch_fwd_loss += forward_loss.item()
                epoch_bwd_loss += backward_loss.item()
                n_batches += 1

                # Compute norms
                v_forward_norm = v_forward.detach().norm(dim=-1).mean().item()
                v_backward_norm = v_backward.detach().norm(dim=-1).mean().item()

                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'fwd': f'{forward_loss.item():.4f}',
                    'bwd': f'{backward_loss.item():.4f}'
                })

                if wandb.run is not None:
                    wandb.log({
                        "finetune/loss": loss.item(),
                        "finetune/fwd_loss": forward_loss.item(),
                        "finetune/bwd_loss": backward_loss.item(),
                        "finetune/v_forward_norm": v_forward_norm,
                        "finetune/v_backward_norm": v_backward_norm,
                        "lr": scheduler.get_last_lr()[0],
                        "finetune/alpha" : alpha.item(),
                        "finetune/beta" : beta.item(),
                        "finetune/lambda" : lam,
                        "epoch": epoch
                    }, step=self.global_step)

                    if self.global_step % print_every == 0 and fixed_test_batch is not None : 
                            fig = self.plot_progress(fixed_test_batch, title=f"Finetune Step {self.global_step}")
                            wandb.log({"finetune/samples": wandb.Image(fig)}, step=self.global_step)
                            os.system("nvidia-smi")

                self.global_step += 1

            if epoch % save_every_epoch == 0 :
                self.save_checkpoint(
                        f"checkpoints/finetune/step_{self.global_step}.pt",
                        optimizer=optimizer,
                        scheduler=scheduler,
                        sb_scheduler=sb_scheduler,     
                        ratio_scheduler=ratio_scheduler, 
                        epoch=epoch
                    )

            self.loss_history['finetune_total'].append(epoch_loss / n_batches)
            self.loss_history['finetune_forward'].append(epoch_fwd_loss / n_batches)
            self.loss_history['finetune_backward'].append(epoch_bwd_loss / n_batches)

    def save_checkpoint(self, filepath, optimizer = None , scheduler = None,sb_scheduler=None, ratio_scheduler =None , epoch=0, max_keep=3):
        """Save model weights and EMA parameters, keeping only last max_keep checkpoints"""
        checkpoint = {
            'model_state_dict': self.v_theta.state_dict(),
            'ema_model_state_dict': self.ema_model.state_dict(), 
            'loss_history': self.loss_history,
            'global_step': self.global_step,      
            'epoch': epoch,       
        }
        if optimizer is not None:
            checkpoint['optimizer_state_dict'] = optimizer.state_dict()   
        if scheduler is not None:
            checkpoint['scheduler_state_dict'] = scheduler.state_dict() 

        if sb_scheduler is not None:
            checkpoint['sb_scheduler_state'] = {
                'step': sb_scheduler.step,
            }
        if ratio_scheduler is not None:
            checkpoint['ratio_scheduler_state'] = {
                'current_step': ratio_scheduler.current_step,
                'phi': ratio_scheduler.phi.cpu(),
            }
            
        torch.save(checkpoint, filepath)
        print(f"Checkpoint saved to {filepath}")
    
        # Keep only the last max_keep checkpoints
        checkpoint_dir = os.path.dirname(filepath)
        checkpoint_name = os.path.basename(filepath)
        
        all_ckpts = sorted(
            [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')],
            key=lambda f: os.path.getmtime(os.path.join(checkpoint_dir, f))
        )
        
        while len(all_ckpts) > max_keep:
            oldest = os.path.join(checkpoint_dir, all_ckpts.pop(0))
            os.remove(oldest)
            print(f"Deleted old checkpoint: {oldest}")
            
    def load_checkpoint(self, filepath, optimizer=None, scheduler=None , sb_scheduler= None , ratio_scheduler = None , back_up_phi = None):
        """Load model weights and EMA parameters"""        
        checkpoint = torch.load(filepath, map_location=self.device)
        state_dict = checkpoint['model_state_dict']     
        self.v_theta.load_state_dict(state_dict)
        self.loss_history = checkpoint['loss_history']
        # self.ema_params = checkpoint['ema_params']
        self.ema_model.load_state_dict(checkpoint['ema_model_state_dict'])

        self.global_step = checkpoint.get('global_step', 0)
        start_epoch = checkpoint.get('epoch', 0)

        if optimizer is not None and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scheduler is not None and 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        if sb_scheduler is not None : 
            if 'sb_scheduler_state' in checkpoint:
                sb_scheduler.step = checkpoint['sb_scheduler_state']['step']
                print(f"Restored SBScheduler at step {sb_scheduler.step}")
                
        if ratio_scheduler is not None :
            if 'ratio_scheduler_state' in checkpoint:
                ratio_scheduler.current_step = checkpoint['ratio_scheduler_state']['current_step']
                ratio_scheduler.phi = checkpoint['ratio_scheduler_state']['phi'].to(self.device)
                print(f"Restored OscillatingSnapScheduler at step {ratio_scheduler.current_step}")
    
        print(f"Checkpoint loaded from {filepath} | Resuming from epoch {start_epoch}, step {self.global_step}")
        return start_epoch + 1

    def plot_progress(self, fixed_source_samples, title="Progress"):
        """
        fixed_source_samples: tensor of shape [B, D] (e.g., EMNIST letters)
        """
        model = self.get_ema_model() 
        model.eval()
        
        sde_samples = fixed_source_samples
        ode_samples = fixed_source_samples
        B = fixed_source_samples.shape[0]

        with torch.no_grad():
            # SDE sampling
            traj_sde = self.sample(sde_samples, method="sde", num_steps=30)
            final_sde = traj_sde[:, -1, :]
            # ODE sampling
            traj_ode = self.sample(ode_samples, method="ode", num_steps=30)
            final_ode = traj_ode[:, -1, :]
        
        def unnorm(x):
            return (x * 0.5 + 0.5).clamp(0, 1)
        
        # number per row
        grid_nrow = 4

        sde_input_grid = torchvision.utils.make_grid(unnorm(sde_samples.reshape(B, 1, 32, 32))[:16],nrow=grid_nrow,padding=2)
        sde_output_grid = torchvision.utils.make_grid(unnorm(final_sde.reshape(B, 1, 32, 32))[:16],nrow=grid_nrow,padding=2)
        ode_input_grid = torchvision.utils.make_grid(unnorm(ode_samples.reshape(B, 1, 32, 32))[:16],nrow=grid_nrow,padding=2)
        ode_output_grid = torchvision.utils.make_grid(unnorm(final_ode.reshape(B, 1, 32, 32))[:16],nrow=grid_nrow,padding=2)
        # Plot
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        axes[0, 0].imshow(sde_input_grid.permute(1, 2, 0).cpu().numpy())
        axes[0, 0].set_title("Input (Fixed Source) - SDE")
        axes[0, 0].axis('off')
        axes[0, 1].imshow(sde_output_grid.permute(1, 2, 0).cpu().numpy())
        axes[0, 1].set_title(f"{title} Output - SDE")
        axes[0, 1].axis('off')
        
        axes[1, 0].imshow(ode_input_grid.permute(1, 2, 0).cpu().numpy())
        axes[1, 0].set_title("Input (Fixed Source) - ODE")
        axes[1, 0].axis('off')
        axes[1, 1].imshow(ode_output_grid.permute(1, 2, 0).cpu().numpy())
        axes[1, 1].set_title(f"{title} Output - ODE")
        axes[1, 1].axis('off')

        plt.show()
        plt.close(fig)
        return fig


    def sample_noisy(self, x0, direction="forward", num_steps=None):
        model = self.get_ema_model()
        model.eval()

        if num_steps is None: num_steps = self.steps

        delta_t = self.T / num_steps
        t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, num_steps + 1)]

        dir_idx = torch.ones(x0.shape[0], device=self.device)

        x = x0.clone()
        trajectory = [x.clone()]
        with torch.no_grad():
            for i, t in enumerate(t_list[:-1]):
                t_tensor = t.expand(x.shape[0], 1)
                drift = model(x, t_tensor, dir_idx)
                # x = x + delta_t * drift + np.sqrt(self.eps * delta_t) * torch.randn_like(x)
                if i < num_steps -1 :
                    x = x + delta_t * drift + np.sqrt(self.eps * delta_t) * torch.randn_like(x)
                else :
                    x = x + delta_t * drift 
                
                trajectory.append(x.clone())
        return torch.stack(trajectory, dim=1)

    def sample_ode(self, x0, direction="forward", num_steps=None):
        model = self.get_ema_model()
        model.eval()

        if num_steps is None: num_steps = self.steps

        delta_t = self.T / num_steps
        t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, num_steps + 1)]

        ones = torch.ones(x0.shape[0], device=self.device)
        zeros = torch.zeros(x0.shape[0], device=self.device)

        x = x0.clone()
        trajectory = [x.clone()]

        with torch.no_grad():
            for i, t in enumerate(t_list[:-1]):
                t_tensor = t.expand(x.shape[0], 1)

                drift_f = model(x, t_tensor, ones)
                drift_b = model(x, 1.0 - t_tensor, zeros)                 
                drift_ode = 0.5 * (drift_f - drift_b)
                
                x = x + delta_t * drift_ode
                trajectory.append(x.clone())

        return torch.stack(trajectory, dim=1)

    def sample(self, x0, method='ode', direction=None, num_steps=None):
        if method == 'ode':
            return self.sample_ode(x0, num_steps=num_steps)
        else:
            return self.sample_noisy(x0, direction=direction, num_steps=num_steps)


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# DATASETS :

In [ ]:
# ==================== MNIST/EMNIST ====================

def load_mnist_emnist(subset_size=5000, only_some_letters=True, seed=42):
    """Load and prepare MNIST and EMNIST datasets (resized to 32x32)"""

    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
        transforms.Lambda(lambda x: x.flatten()),
    ])

    mnist_dataset = torchvision.datasets.MNIST(
        root='./data', train=True, download=True, transform=transform
    )

    if only_some_letters:
        emnist_dataset = torchvision.datasets.EMNIST(
            root='./data', split='byclass', train=True, download=True, transform=transform
        )
        target_classes = [10, 11, 12, 13, 14, 36, 37, 38, 39, 40]
        mask = torch.isin(emnist_dataset.targets, torch.tensor(target_classes))
        
        emnist_dataset.data = emnist_dataset.data[mask].permute(0, 2, 1)
        emnist_dataset.targets = emnist_dataset.targets[mask]
        
        class_mapping = {old_label: new_label for new_label, old_label in enumerate(target_classes)}
        emnist_dataset.targets = torch.tensor([class_mapping[t.item()] for t in emnist_dataset.targets])
    else:
        emnist_dataset = torchvision.datasets.EMNIST(
            root='./data', split='letters', train=True, download=True, transform=transform
        )
        emnist_dataset.data = emnist_dataset.data[mask].permute(0, 2, 1)

    # manual seed
    g = torch.Generator()
    g.manual_seed(seed)

    mnist_size = min(subset_size, len(mnist_dataset))
    emnist_size = min(subset_size, len(emnist_dataset))

    # Generate random permutations of indices using the seeded generator
    mnist_indices = torch.randperm(len(mnist_dataset), generator=g)[:mnist_size]
    emnist_indices = torch.randperm(len(emnist_dataset), generator=g)[:emnist_size]

    mnist_data = torch.stack([mnist_dataset[i][0] for i in mnist_indices])
    emnist_data = torch.stack([emnist_dataset[i][0] for i in emnist_indices])
    return mnist_data.numpy(), emnist_data.numpy()

# MNIST-EMNIST experiments :

In [ ]:
####################### Hyperparameters
project_name = "mnist-schrodinger-bridge"
run_name = "run_name"

pretrain_epochs = 213     
finetune_epochs = 60 #320      
batch_size = 128          
eps = 1.0                   

pretrain_lr = 1.5e-4  
finetune_lr = 1e-4

decay = 0.999      
steps = 30                  
num_workers = 2
base_channels = 128

ratio_osc = 0.75
ratio_leap = 0.05 

seed = 42

####################### Initialize W&B
# if wandb.run is not None:
#     wandb.finish()
    
# run = wandb.init(
#     project=project_name,
#     name=run_name,
#     config={
#         "eps": eps,
#         "pretrain_lr": pretrain_lr,
#         "finetune_lr": finetune_lr,
#         "batch_size": batch_size,
#         "pretrain_epochs": pretrain_epochs,
#         "finetune_epochs": finetune_epochs,
#         "steps": steps,
#         "decay": decay,
#         "num_workers": num_workers,
#         "base_channels" : base_channels,
#         "ratio_osc" : ratio_osc,
#         "ratio_leap" : ratio_leap
#     }
# ) 
# run = wandb.init(project=project_name, id='j3bm4x4m', resume="must")

# RUN_ID = run.id
# print('RUN_ID', RUN_ID)

print("Loading MNIST and EMNIST...")
mnist_data, emnist_data = load_mnist_emnist(subset_size=62000 , seed = seed)

split_idx = 60000
mnist_train = mnist_data[:split_idx]
emnist_train = emnist_data[:split_idx]
mnist_val = mnist_data[split_idx:]
emnist_val = emnist_data[split_idx:]

print(f"Training Samples: {len(mnist_train)}")
print(f"Validation Samples: {len(mnist_val)}")

# Laws
law_0 = mnist_train
law_1 = emnist_train

# seed
set_global_seed(seed)

# model
bridge = Schrodinger_Bridge_Matching(
    Law_0=law_0,
    Law_1=law_1,
    pretrain_epochs=pretrain_epochs, 
    finetune_epochs=finetune_epochs,
    Bs=batch_size,
    steps=steps,
    eps=eps,
    pretrain_lr=pretrain_lr,
    finetune_lr=finetune_lr,
    decay=decay,  
    num_workers=num_workers,
    base_channels = base_channels,
    ratio_osc = ratio_osc,
    ratio_leap = ratio_leap,
)

========================================
# We take the first 16 letters for val
fixed_val_digits = torch.tensor(mnist_train[:16], dtype=torch.float32).to(bridge.device)

output_folder = '/kaggle/working/bridge_checkpoints'
os.makedirs(output_folder, exist_ok=True)

# # ##--- PHASE 1: PRETRAINING ---
print("\n=== Phase 1: Pretraining ===")
bridge.pretrain_bridge(print_every=1000, save_every_epoch = 20, resume_from=None , fixed_test_batch=fixed_val_digits)
pretrained_path = f'{output_folder}/Pretrained_bridge__{RUN_ID}.pt'
bridge.save_checkpoint(pretrained_path)
print(f"Saved: {pretrained_path}")

# # SAFETY ZIP 1
shutil.make_archive('/kaggle/working/backup_phase1_pretrained', 'zip', output_folder)
print("SAFE: Phase 1 zip created.")

##--- PHASE 2: FINETUNING ---
print("\n=== Phase 2: Finetuning ===")
pretrained_path = "/kaggle/input/models/sleeper00s/shrodinger-bridge-mnist-5m-pretrained/pytorch/psukwu75/1/Pretrained_bridge__psukwu75.pt"
bridge.load_checkpoint(pretrained_path)

bridge.finetune_bridge(print_every=1000, save_every_epoch = 5 ,resume_from="/kaggle/input/models/sleeper00s/shrodinger-bridge-mnist-5m-finetuned-30-epochs-75p/pytorch/finteuned_5m_3jvli0u8_step_112091.pt/1/Finteuned_5M_3jvli0u8_step_112091.pt", fixed_test_batch=fixed_val_digits)
finetuned_path = f'{output_folder}/Finetuned_{RUN_ID}.pt'
bridge.save_checkpoint(finetuned_path)
print(f"Saved: {finetuned_path}")

# SAFETY ZIP 2
shutil.make_archive('/kaggle/working/backup_final_all', 'zip', output_folder)
print("SAFE: Final zip created.")
wandb.finish()

Loading MNIST and EMNIST...


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 515kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.75MB/s]
100%|██████████| 562M/562M [00:01<00:00, 288MB/s] 


Training Samples: 60000
Validation Samples: 0
device cuda


#  Evaluation Pipeline :

In [ ]:
import shutil
import tempfile
from pathlib import Path
import torchvision
import torchvision.transforms as transforms
from PIL import Image
# pip install clean-fid

from cleanfid import fid


# ─── Data loading ─────────────────────────────────────────────────────────────

def get_emnist_test(n_samples: int = 4000, seed: int = 42) -> torch.Tensor:
    """
    Returns n_samples EMNIST test images from the same class subset used in training.
    Resized to 32x32, normalised to [-1, 1], flattened → (N, 1024).
    """
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
        transforms.Lambda(lambda x: x.flatten()),
    ])

    dataset = torchvision.datasets.EMNIST(
        root="./data", split="byclass", train=False,
        download=True, transform=transform,
    )

    target_classes = [10, 11, 12, 13, 14, 36, 37, 38, 39, 40]
    mask            = torch.isin(dataset.targets, torch.tensor(target_classes))
    dataset.data    = dataset.data[mask].permute(0, 2, 1)   # fix EMNIST transpose quirk
    dataset.targets = dataset.targets[mask]

    class_mapping   = {old: new for new, old in enumerate(target_classes)}
    dataset.targets = torch.tensor([class_mapping[t.item()] for t in dataset.targets])

    g       = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=g)[:n_samples]
    return torch.stack([dataset[i][0] for i in indices])    # (N, 1024)


def get_mnist_train_all() -> torch.Tensor:
    """Returns all 60K MNIST training images, flattened → (60000, 1024)."""
    transform = transforms.Compose([
        transforms.Resize(32),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
        transforms.Lambda(lambda x: x.flatten()),
    ])
    dataset = torchvision.datasets.MNIST(
        root="./data", train=True, download=True, transform=transform,
    )
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=512, shuffle=False, num_workers=4,
    )
    return torch.cat([imgs for imgs, _ in loader], dim=0)   # (60000, 1024)


# ─── Helpers ──────────────────────────────────────────────────────────────────

def tensor_to_png_folder(tensors: torch.Tensor, folder: Path, img_size: int = 32) -> None:
    """
    Saves flat tensors (N, C*H*W) in [-1, 1] as RGB PNGs.
    clean-fid reads images from disk, so we dump everything to a temp folder.
    """
    folder.mkdir(parents=True, exist_ok=True)
    imgs = tensors.view(-1, 1, img_size, img_size)
    imgs = ((imgs + 1.0) / 2.0 * 255).clamp(0, 255).byte()
    for i, img in enumerate(imgs):
        pil_img = Image.fromarray(img.squeeze(0).numpy(), mode="L").convert("RGB")
        pil_img.save(folder / f"{i:05d}.png")


def compute_msd(original: torch.Tensor, generated: torch.Tensor) -> float:
    """Mean Squared Distance between paired flat tensors (N, D) in [-1, 1]."""
    assert original.shape == generated.shape, "Shape mismatch between input/output pairs."
    return ((original - generated) ** 2).mean(dim=1).mean().item()


# ─── Core evaluation ──────────────────────────────────────────────────────────

# Load MNIST reference once — shared across all checkpoint evaluations.
_mnist_train_cache: torch.Tensor | None = None

def evaluate(
    bridge_model,
    device:     torch.device,
    num_steps:  int = 30,
    n_samples:  int = 4000,
    batch_size: int = 128,
) -> dict:
    """
    Evaluates a single bridge model checkpoint.

    bridge_model must expose:
        .sample(x0, method, direction, num_steps) → trajectory (B, T+1, D)
    The final time step [:, -1, :] is taken as the generated output.

    Returns:
        {"FID": float, "MSD": float}
    """
    global _mnist_train_cache

    # ── 1. EMNIST test inputs ─────────────────────────────────────────────────
    print("  Loading EMNIST test data ...")
    emnist_test = get_emnist_test(n_samples=n_samples)      # (N, 1024)

    # ── 2. Generate MNIST samples ─────────────────────────────────────────────
    print(f"  Generating {n_samples} MNIST samples ...")
    chunks = []

    with torch.no_grad():
        for start in range(0, n_samples, batch_size):
            x0   = emnist_test[start : start + batch_size].to(device)
            traj = bridge_model.sample(
                x0, method="noisy", direction="forward", num_steps=num_steps,
            )
            chunks.append(traj[:, -1, :].cpu())             # final time step only

    generated = torch.cat(chunks, dim=0)                    # (N, 1024)

    # ── 3. MSD ────────────────────────────────────────────────────────────────
    msd = compute_msd(emnist_test, generated)
    print(f"  MSD : {msd:.4f}")

    # ── 4. FID via clean-fid ──────────────────────────────────────────────────
    # Cache MNIST train set so we only load it once across all checkpoints.
    if _mnist_train_cache is None:
        print("  Loading full MNIST train set (cached for subsequent runs) ...")
        _mnist_train_cache = get_mnist_train_all()

    tmp_root = Path(tempfile.mkdtemp())
    real_dir = tmp_root / "real"
    fake_dir = tmp_root / "fake"

    try:
        print("  Saving reference images to disk ...")
        tensor_to_png_folder(_mnist_train_cache, real_dir)

        print("  Saving generated images to disk ...")
        tensor_to_png_folder(generated, fake_dir)

        print("  Computing FID ...")
        fid_score = fid.compute_fid(
            str(real_dir),
            str(fake_dir),
            mode="clean",   # standardised, paper-reproducible mode
            num_workers=4,
        )
        print(f"  FID : {fid_score:.4f}")

    finally:
        shutil.rmtree(tmp_root)     # always clean up, even on crash

    return {"FID": fid_score, "MSD": msd}


# ─── Checkpoint loop ──────────────────────────────────────────────────────────

# ── Config ───────────────────────────────────────────────────────────────
num_steps  = 30
n_samples  = 4000
batch_size = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Format: ["path/to/checkpoint.pt", "Model Name"]
checkpoints = [
    ["/kaggle/input/models/sleeper00s/to-evaluation-5m-shrodinger-bridge-mnist/pytorch/default/1/Pretrained_bridge__psukwu75.pt",          "Pretrained"],
    ["/kaggle/input/models/sleeper00s/to-evaluation-5m-shrodinger-bridge-mnist/pytorch/default/1/Finteuned_5M_3jvli0u8_step_112091.pt",   "Finetuning Start"],
    ["/kaggle/input/models/sleeper00s/to-evaluation-5m-shrodinger-bridge-mnist/pytorch/default/1/STEP_11.PT",                              "Finetuning 11"],
    ["/kaggle/input/models/sleeper00s/to-evaluation-5m-shrodinger-bridge-mnist/pytorch/default/1/STEP_12.PT",                              "Finetuning 12"],
    ["/kaggle/input/models/sleeper00s/to-evaluation-5m-shrodinger-bridge-mnist/pytorch/default/1/STEP_13.PT",                              "Finetuning 13"],
]

# ── Loop ─────────────────────────────────────────────────────────────────
all_results = {}

for entry in checkpoints:
    # Support both ["path,Name"] and ["path", "Name"] formats
    path, name = (
        [s.strip() for s in entry[0].split(",", 1)]
        if len(entry) == 1 else entry
    )

    print(f"\n{'='*60}")
    print(f"  Evaluating : {name}")
    print(f"  Checkpoint : {path}")
    print(f"{'='*60}")

    bridge.load_checkpoint(path)

    results = evaluate(
        bridge_model=bridge,
        device=device,
        num_steps=num_steps,
        n_samples=n_samples,
        batch_size=batch_size,
    )
    
    all_results[name] = results
    print(f"  → FID: {results['FID']:.4f}  |  MSD: {results['MSD']:.4f}")

# ── Summary table ────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"{'Model':<35} {'FID':>8} {'MSD':>8}")
print(f"{'-'*60}")
for name, res in all_results.items():
    print(f"{name:<35} {res['FID']:>8.4f} {res['MSD']:>8.4f}")
print(f"{'='*60}")